In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [0]:
query = """
SELECT 
    fs.order_date,
    fs.product_fk,
    fs.territory_fk,
    fs.order_quantity,
    fs.unit_price,
    fs.total_due,
    dp.product_name,
    dp.product_category_name,
    dt.country_region_name,
    ds.store_name,
    ds.business_entity_id as store_id
FROM ted_dev.marts.fact_sales fs
JOIN ted_dev.marts.dim_product dp ON fs.product_fk = dp.product_pk
JOIN ted_dev.marts.dim_territory dt ON fs.territory_fk = dt.territory_pk
LEFT JOIN ted_dev.marts.dim_store ds ON fs.sales_person_fk = ds.sales_person_id
ORDER BY fs.order_date
"""

df = spark.sql(query).toPandas()
df['order_date'] = pd.to_datetime(df['order_date'])

print(f"Dados carregados: {len(df):,} registros")
print(f"Período: {df['order_date'].min()} a {df['order_date'].max()}")
print(f"Produtos únicos: {df['product_fk'].nunique()}")
print(f"Lojas distintas: {df['store_id'].nunique()}")

monthly_data = df.groupby([
    'product_fk', 
    'store_id',
    pd.Grouper(key='order_date', freq='M')
]).agg({
    'order_quantity': 'sum',
    'unit_price': 'mean',
    'total_due': 'sum',
    'product_name': 'first',
    'store_name': 'first',
    'country_region_name': 'first'
}).reset_index()

monthly_data = monthly_data.dropna(subset=['store_id'])


# Estimativa de zíperes para produção de luvas

### Objetivo
Desenvolver estimativa da quantidade de zíperes necessários para os próximos 3 meses, considerando a proporção técnica de 2 zíperes por par de luvas.

In [0]:
gloves_data = monthly_data[
    monthly_data['product_name'].str.startswith('Half-Finger Gloves', na=False)
].copy()

monthly_totals = gloves_data.groupby(
    pd.Grouper(key='order_date', freq='M')
)['order_quantity'].sum().sort_index()

if len(monthly_totals) >= 3:
    last_3_months_avg_gloves = monthly_totals.tail(3).mean()
    projected_3_months_gloves = last_3_months_avg_gloves * 3
    zippers_needed_ma3 = projected_3_months_gloves * 2
    zippers_per_month_ma3 = last_3_months_avg_gloves * 2  # Zíperes por mês

if len(monthly_totals) >= 12:
    last_12_months_avg_gloves = monthly_totals.tail(12).mean()
    projected_3_months_gloves_12m = last_12_months_avg_gloves * 3
    zippers_needed_ma12 = projected_3_months_gloves_12m * 2
    zippers_per_month_ma12 = last_12_months_avg_gloves * 2  # Zíperes por mês


print("Média dos últimos 3 meses:")

print(f"- Média mensal: {last_3_months_avg_gloves:,.0f} luvas/mês")
print(f"- Zíperes por mês: {zippers_per_month_ma3:,.0f} unidades/mês")
print(f"- Previsão para os próximos 3 meses: {projected_3_months_gloves:,.0f} luvas")
print(f"- Total de zíperes para 3 meses: {zippers_needed_ma3:,.0f} unidades")

print("Média dos últimos 12 meses:")

print(f"- Média anual: {last_12_months_avg_gloves:,.0f} luvas/mês")
print(f"- Zíperes por mês: {zippers_per_month_ma12:,.0f} unidades/mês")
print(f"- Previsão para os próximos 3 meses: {projected_3_months_gloves_12m:,.0f} luvas")
print(f"- Total de zíperes para 3 meses: {zippers_needed_ma12:,.0f} unidades")

In [0]:
historical_zippers = monthly_totals * 2

sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(1, 1, figsize=(18, 9))
fig.suptitle('Estimativa e histórico da demanda de zíperes', fontsize=24, fontweight='bold')

ax.plot(historical_zippers.index, historical_zippers.values, 'o-', color='gray', alpha=0.7, label='Demanda histórica')

for date, value in historical_zippers.items():
    ax.text(date, value + 500, f'{value:,.0f}', ha='center', va='bottom', fontsize=9, alpha=0.8)

future_dates = pd.date_range(start=historical_zippers.index[-1], periods=4, freq='M')[1:]

if zippers_per_month_ma3 is not None:
    ax.plot(future_dates, [zippers_per_month_ma3] * 3, 'o--', color='coral', lw=2, 
            label=f'Previsão 3M: {zippers_per_month_ma3:,.0f}/mês')
    ax.text(future_dates[-1], zippers_per_month_ma3, f' {zippers_per_month_ma3:,.0f}/mês', color='coral',
            ha='left', va='center', fontsize=12, fontweight='bold')
    
    ax.text(future_dates[1], zippers_per_month_ma3 + 2000, 
            f'Total 3 meses: {zippers_needed_ma3:,.0f}', 
            color='coral', ha='center', va='bottom', fontsize=11, fontweight='bold',
            bbox=dict(boxstyle="round,pad=0.3", facecolor='white', edgecolor='coral', alpha=0.8))

if zippers_per_month_ma12 is not None:
    ax.plot(future_dates, [zippers_per_month_ma12] * 3, 'o--', color='steelblue', lw=2, 
            label=f'Previsão 12M: {zippers_per_month_ma12:,.0f}/mês')
    ax.text(future_dates[-1], zippers_per_month_ma12, f' {zippers_per_month_ma12:,.0f}/mês', color='steelblue',
            ha='left', va='center', fontsize=12, fontweight='bold')
    
    ax.text(future_dates[1], zippers_per_month_ma12 - 2000, 
            f'Total 3 meses: {zippers_needed_ma12:,.0f}', 
            color='steelblue', ha='center', va='top', fontsize=11, fontweight='bold',
            bbox=dict(boxstyle="round,pad=0.3", facecolor='white', edgecolor='steelblue', alpha=0.8))

ax.axvline(historical_zippers.index[-1], color='black', linestyle=':', lw=2, label='Início da previsão')
ax.set_title('Histórico mensal e projeção para os próximos 3 meses', fontsize=16)
ax.set_ylabel('Quantidade de zíperes (mensal)')
ax.set_xlabel('Período')
ax.legend(loc='upper left', fontsize=11)
ax.set_ylim(bottom=0)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()